# 01 — Data & EDA: *Is rating even visible in one game?*

**Project.** Estimate a Lichess player's **blitz** Elo from *how they played a single game* — a
regression problem with a per-game uncertainty interval, plus rating bands for a confusion matrix.
This first notebook establishes the **data story** and asks the question the entire project hinges on:
*does a single blitz game carry a legible signal about the player's strength at all?*

**What this notebook covers (rubric: Data, EDA).**
1. The ingest → clean → features **data funnel**, and the selection bias baked into it.
2. The **label**: rating distribution and the derived rating bands.
3. **Games per player** — why the median is 1, and why that fact drives the whole design.
4. Six EDA probes of *signal vs. noise*: move quality, phase blunders, time usage, book depth,
   the opponent-rating leak, and a feature↔rating correlation map.

**The tension we are going to surface.** Move-quality features track rating *on average* remarkably
well — yet the per-game spread is enormous. That gap between a strong *aggregate* trend and a noisy
*single-game* observation is the central difficulty this project is built around, and it reappears in
every later notebook (the ±240-Elo interval, the tail bias, the aggregation curve).

> **Conventions.** Every deck figure is written to this notebook's folder
> `reports/figures/notebook_01_eda/` via `src.plotting.save_fig` and then displayed from disk, so the
> notebook renders identically whether or not an interactive backend is active, and the slide deck
> pulls the exact same PNGs. (Figures produced by the pipeline itself — `src.evaluate` — live in the
> `reports/figures/` root; the per-notebook subfolders hold figures authored in the notebooks.) All
> modelling logic lives in `src/`; cells here only *call* it or read the processed parquets —
> *notebooks are for looking.*

In [ ]:
# --- Setup: paths, style, and the shared save-and-show helper -------------------------------
import sys, warnings
from pathlib import Path

# Make `import src...` work whether the notebook is run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from src.config import load_config
from src.plotting import set_style, save_fig

warnings.filterwarnings("ignore")          # keep pandas/matplotlib chatter out of the deck
cfg = load_config()
set_style()                                 # one house style for every figure

PROC = Path(cfg["paths"]["processed"])      # absolute, from config — CWD-independent
FIG = Path(cfg["paths"]["figures"]) / "notebook_01_eda"   # this notebook's figure folder (auto-created)
BANDS = cfg["rating_bands"]                  # [0, 1200, 1400, 1600, 1800, 2000, 3000]

def show(path):
    "Display a figure that save_fig() just wrote to reports/figures/."
    display(Image(filename=str(path)))

print("processed dir:", PROC)
print("rating bands  :", BANDS)

## 1. The data funnel

The pipeline turns one ~30 GB monthly Lichess dump into a modelling table through five stages
(`ingest → clean → features → model → evaluate`), each handing the next a parquet on disk. The
funnel below is dominated by one filter: **we keep only games that already carry a stored Stockfish
`[%eval]` annotation.** Those are games a player *opted to have analysed*, roughly 9% of blitz
games — and emphatically **not** a random sample.

The counts on the *scanned* side come from the ingest/clean run over the full month (the 30 GB dump
is not re-scanned in this notebook); the *processed* side is computed live from the parquets shipped
on the shared drive.

In [ ]:
# Load the processed tables. games_clean carries the raw movetext (~300 MB); we only need a few
# columns here, so we read a projection to keep memory light.
gc_cols = ["game_id", "white", "black", "white_elo", "black_elo", "result", "eco", "time_control"]
games = pd.read_parquet(PROC / "games_clean_300k.parquet", columns=gc_cols)
feats = pd.read_parquet(PROC / "features_300k_v2.parquet")

n_games = len(games)
n_instances = len(feats)          # one row per (game, side) — the actual modelling examples
n_players = feats["username"].nunique()
print(f"games_clean : {n_games:,} games")
print(f"features    : {n_instances:,} instances (2 per game)  x  {feats.shape[1]} columns")
print(f"players     : {n_players:,} distinct usernames")

In [ ]:
# Documented ingest/clean funnel over the full 2025-05 dump (from the pipeline run logs), followed
# by the live-computed processed counts. The big drop is the stored-eval filter.
funnel = pd.DataFrame([
    ("raw games scanned",              15_000_000, "full 2025-05 standard-rated stream"),
    ("blitz time control",              6_980_000, "~47% — base+40*inc in 180..479 s"),
    ("carry a stored [%eval]",            648_594, "9.3% of blitz — the analysable pool (selection bias)"),
    ("clean (dedup + valid labels)",      647_983, "drop aborts, bad Elo, disallowed terminations"),
    ("sampled for modelling (games)",     300_000, "reservoir sample, fixed seed"),
    ("+ player-cohort games",               6_117, "all games of a hash-sampled player subset"),
], columns=["stage", "count", "note"])

processed = pd.DataFrame([
    ("processed games_clean (sample + cohort)", n_games,     "deduped, movetext retained"),
    ("instances = games x 2 sides",             n_instances, "per-player rows — the model examples"),
    ("distinct players",                        n_players,   "the grouped-split key"),
], columns=["stage", "count", "note"])

print("INGEST / CLEAN FUNNEL (full-dump logs)")
display(funnel)
print("\nPROCESSED (live from parquet)")
display(processed)

**Reading the funnel.** Two facts to carry forward:

- **Selection bias is structural, not incidental.** The eval'd subset over-represents games players
  found interesting enough to analyse — often decisive, sharp, or instructive games. A rating
  estimator trained here is calibrated to *that* population, and may not transfer to a uniform random
  sample of blitz games. This is a documented limitation, not a bug we can filter away (the raw dump
  strips the information we would need to correct it). Validating against self-computed fixed-depth
  evals on a random sample is on the backlog.
- **Each game becomes two examples.** `clean` explodes every game into two **instances**, one per
  side, each labelled with *that* player's own rating. The opponent's rating and username are
  dropped by construction — the leakage guard we motivate empirically in §8 below.

## 2. The label: rating distribution and bands

The regression target is the player's blitz Elo. The distribution is the familiar rating bell:
concentrated in the 1400–1900 range, with sparse tails. Those sparse tails are the first hint of the
tail-bias problem that notebook 05 dissects — an MAE learner sees very few examples out at 2200+ or
below 1000, so it has little incentive to predict them.

In [ ]:
r = feats["rating"].to_numpy()
print(f"rating: min {r.min()}  max {r.max()}  mean {r.mean():.0f}  median {np.median(r):.0f}  std {r.std():.0f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r, bins=80, color="#2c6fbb", alpha=0.85, edgecolor="white", linewidth=0.3)
for edge in BANDS[1:-1]:
    ax.axvline(edge, color="#c0504d", ls="--", lw=1, alpha=0.8)
ax.set(xlabel="Blitz rating (Elo)", ylabel="Instances",
       title="Rating distribution with band edges (dashed)")
ax.text(0.99, 0.95, f"mean {r.mean():.0f}\nmedian {np.median(r):.0f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round", fc="white", ec="0.7"))
show(save_fig(fig, "eda_rating_distribution", FIG))

In [ ]:
# Band sizes (the classes for the later confusion matrix). Note how few instances live in the
# extreme bands relative to the centre — this imbalance is what drags tail predictions inward.
labels = [f"{BANDS[i]}-{BANDS[i+1]}" for i in range(len(BANDS) - 1)]
band_idx = np.digitize(r, BANDS[1:-1])
counts = np.bincount(band_idx, minlength=len(labels))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, counts, color="#2c6fbb")
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, c, f"{c/ len(r)*100:.0f}%",
            ha="center", va="bottom", fontsize=9)
ax.set(xlabel="Rating band", ylabel="Instances", title="Band sizes (confusion-matrix classes)")
ax.tick_params(axis="x", rotation=20)
show(save_fig(fig, "eda_band_sizes", FIG))

## 3. Games per player — why the median is 1

A uniform *game* sample contains a long tail of players seen exactly once. That is the single most
important structural fact about the data for this project:

In [ ]:
gpp = feats.groupby("username").size()
print(f"players: {len(gpp):,}   median games/player: {gpp.median():.0f}   mean: {gpp.mean():.2f}   max: {gpp.max()}")
print(f"share of players with >= 2 games: {(gpp >= 2).mean()*100:.1f}%")

# Compare against the dedicated player-cohort sample (all games of a hash-sampled player subset),
# which exists precisely so the aggregation curve has players with many games.
cohort = pd.read_parquet(PROC / "player_cohort.parquet", columns=["game_id", "white", "black"])
cohort_gpp = pd.concat([cohort["white"], cohort["black"]]).value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.hist(np.clip(gpp, 0, 12), bins=np.arange(1, 14) - 0.5, color="#2c6fbb")
ax1.set(xlabel="Games per player (clipped at 12)", ylabel="Players",
        title=f"Uniform game sample (median {gpp.median():.0f})")
ax2.hist(np.clip(cohort_gpp, 0, 20), bins=np.arange(1, 22) - 0.5, color="#3aa66f")
ax2.set(xlabel="Games per player (clipped at 20)", ylabel="Players",
        title=f"Player-cohort sample (mean {cohort_gpp.mean():.1f})")
fig.tight_layout()
show(save_fig(fig, "eda_games_per_player", FIG))

**Why this matters.** If most players appear once, then for most of the dataset "estimate this
player's rating" really means "estimate it from *one* game" — there is no second game to average
away the noise. That is the honest, hard version of the problem, and it is why the headline metric is
a single-game MAE with an interval, not a per-player number. The **player-cohort** sample (right)
keeps every game of a small hashed subset of players, so the aggregation curve in notebook 04 can ask
the complementary question: *how much does the error fall if we DO get more games per player?*

## 4. Move quality vs rating (the core signal)

Now the core EDA question. We bin instances into 100-Elo rating slices and plot the mean of a feature
per slice, with the inter-quartile band shaded. If a feature is informative, its mean should move
monotonically with rating; if it is *usable for a single game*, the IQR band should be *narrow*.
Watch both.

In [ ]:
def rating_curve(df, col, lo=600, hi=2600, step=100, min_n=200):
    "Mean + IQR of `col` in 100-Elo rating slices (presentation helper; skips NaN and thin bins)."
    rr = df["rating"].to_numpy(float)
    vv = df[col].to_numpy(float)
    edges = np.arange(lo, hi + step, step)
    idx = np.digitize(rr, edges)
    cen, mean, q25, q75 = [], [], [], []
    for b in range(1, len(edges)):
        m = (idx == b) & ~np.isnan(vv)
        if m.sum() >= min_n:
            cen.append((edges[b - 1] + edges[b]) / 2)
            mean.append(np.mean(vv[m])); q25.append(np.percentile(vv[m], 25)); q75.append(np.percentile(vv[m], 75))
    return map(np.array, (cen, mean, q25, q75))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, col, ylab, title, color in [
    (ax1, "cpl_mean", "Mean centipawn loss", "Move error vs rating", "#c0504d"),
    (ax2, "acc_mean", "Lichess Accuracy%", "Accuracy vs rating", "#3aa66f"),
]:
    c, m, lo, hi = rating_curve(feats, col)
    ax.fill_between(c, lo, hi, color=color, alpha=0.18, label="inter-quartile range")
    ax.plot(c, m, "-o", color=color, ms=4, label="slice mean")
    ax.set(xlabel="Rating (Elo)", ylabel=ylab, title=title)
    ax.legend(fontsize=9)
fig.tight_layout()
show(save_fig(fig, "eda_quality_vs_rating", FIG))

**Reading it.** The slice means are almost textbook-monotone: mean centipawn loss falls from ~77 in
the 0–1200 band to ~43 above 2000 — a clean, near-two-fold separation — while Accuracy% climbs in
lockstep from ~86% to ~91%. *This is the signal that makes the project possible at all.*

But look at the shaded IQR bands: they are **wide and heavily overlapping** between adjacent slices.
A single game with a mean CPL of, say, 55 is entirely consistent with a 1400 player having a clean
game or an 1800 player having a scrappy one. The *average* separates the bands cleanly; a *single
game* does not. That is the signal-vs-noise tension in one picture — and it is precisely why one
game buys only a wide interval (notebook 04).

## 5. Blunder rate by phase

Move quality is not uniform across the game. Splitting the blunder rate (moves that throw ≥0.30 of
winning chances) into opening / middlegame / endgame reveals *where* strength actually separates
players — and motivates the per-phase feature design.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
for col, name, color in [
    ("opening_blunder_rate", "opening", "#2c6fbb"),
    ("middlegame_blunder_rate", "middlegame", "#e07b39"),
    ("endgame_blunder_rate", "endgame", "#8064a2"),
]:
    c, m, _, _ = rating_curve(feats, col)
    ax.plot(c, m * 100, "-o", ms=4, color=color, label=name)
ax.set(xlabel="Rating (Elo)", ylabel="Blunder rate (% of moves)",
       title="Blunder rate by game phase vs rating")
ax.legend(title="phase")
show(save_fig(fig, "eda_blunder_by_phase", FIG))

**Reading it.** The three phases behave very differently, and the differences are informative:

- **Opening blunders fall the most steeply** — from ~8.3% in the lowest band to ~1.9% above 2000, a
  roughly four-fold drop. Opening theory is the great separator: weaker players make concrete,
  detectable opening errors that strong players essentially never make. (This is the same fact that
  makes `opening_acc_mean` the single strongest linear correlate of rating — see §9.)
- **Middlegame carries the highest blunder rate at every level** (~11.6% → ~7.0%). It is the
  sharpest, most complex phase, where even strong players err most often.
- **Endgame blunder rate is nearly flat** (~6.5% → ~5.6%). Endgames are hard for everyone, so an
  endgame blunder discriminates rating only weakly — a useful caution against assuming "later phase =
  more signal."

This is exactly why the feature stage computes move quality **per phase** rather than only overall,
and why the `has_endgame` flag matters: 37.5% of games never reach an endgame, and even when they do,
that phase carries less rating signal than the opening.

## 6. Time management

All sampled games carry a `[%clk]` stream, so time features are broadly observable. They tell a
consistent — if subtler — story than move quality.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, ylab, title in [
    (axes[0], "move_time_mean", "seconds / move", "Mean move time"),
    (axes[1], "fast_move_share", "share of moves", "Fast moves (<1s)"),
    (axes[2], "time_trouble_share", "share of moves", "Time trouble (<15s left)"),
]:
    c, m, _, _ = rating_curve(feats, col)
    y = m if col == "move_time_mean" else m * 100
    ax.plot(c, y, "-o", ms=4, color="#4bacc6")
    ax.set(xlabel="Rating (Elo)", ylabel=(ylab if col == "move_time_mean" else ylab + " (%)"), title=title)
fig.tight_layout()
show(save_fig(fig, "eda_time_vs_rating", FIG))

**Reading it.** Stronger players are **faster and more decisive**: mean move time falls from ~6.5 s
to ~4.6 s across the range, and the share of near-instant (<1 s) moves more than doubles (~5% → ~11%)
as pattern recognition lets them play known positions on sight. Interestingly, `time_trouble_share`
moves the *opposite* way — it rises slightly with rating (~2.5% → ~6%): strong players push their
clocks harder in sharp positions, so they dip under 15 s a little more often. It is the weakest of
the three clock signals. These features are also entangled with the time control itself (a 300+3 game
naturally has higher move times than a 180+0 game), so the model gets the cleanest lift by combining
them with move quality rather than reading them alone — dropping the whole clock group costs ~9 MAE
(notebook 04), less than the engine group but real.

## 7. Book depth: opening vs post-book accuracy

The feature set carries both `opening_acc_mean` (accuracy inside the first ~15 moves) and
`acc_after_book` (accuracy once past the opening). Comparing them by band answers "where does the
accuracy gap between weak and strong players actually live?"

In [ ]:
labels = [f"{BANDS[i]}-{BANDS[i+1]}" for i in range(len(BANDS) - 1)]
feats["_band"] = pd.cut(feats["rating"], BANDS, labels=labels, right=False)
by_band = feats.groupby("_band", observed=True)[["opening_acc_mean", "acc_after_book"]].mean()
print(by_band.round(2))

fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(by_band))
ax.bar(x - 0.2, by_band["opening_acc_mean"], width=0.4, label="opening accuracy", color="#2c6fbb")
ax.bar(x + 0.2, by_band["acc_after_book"], width=0.4, label="post-book accuracy", color="#e07b39")
ax.set_xticks(x); ax.set_xticklabels(by_band.index, rotation=20)
ax.set(xlabel="Rating band", ylabel="Lichess Accuracy%", title="Opening vs post-book accuracy by band")
ax.set_ylim(80, 96); ax.legend()
show(save_fig(fig, "eda_book_depth", FIG))

**Reading it — a mildly counter-intuitive result.** The *opening* is where the accuracy gap is
widest: `opening_acc_mean` fans out from ~85.9% to ~93.0% across bands (a ~7-point spread), whereas
`acc_after_book` barely moves (~86.4% → ~88.7%, a ~2-point spread). In other words, the strongest
univariate signal of rating is how cleanly you navigate *theory*, not how well you improvise
afterwards. Once out of book, positions are sharper and everyone's accuracy compresses, so post-book
accuracy discriminates weakly on its own.

This does **not** make `acc_after_book` useless — a gradient-boosted tree can exploit it in
*interaction* with other features (e.g. "high opening accuracy but a post-book collapse" is a
distinctive pattern) even when its marginal correlation is small. Whether it pulls its weight is a
model question we answer with SHAP and ablations in notebook 04, not an EDA question. The EDA verdict
is simply: opening accuracy is the headline correlate.

## 8. The opponent-rating leak (why matchmaking must be excluded)

One crucial negative result shapes the whole feature design. Lichess pairs players of *similar*
rating, so the opponent's rating is an almost-perfect predictor of yours — and the game result given
the rating gap is highly predictable too. Any feature derived from the opponent would smuggle the
label into the model. We demonstrate the leak here precisely to justify **banning** it.

In [ ]:
gap = (games["white_elo"] - games["black_elo"]).to_numpy(float)
white_win = (games["result"] == "1-0").to_numpy(float)

# P(white wins) as a function of the rating gap, in 50-Elo bins.
edges = np.arange(-400, 401, 50)
idx = np.digitize(gap, edges)
cen, pwin = [], []
for b in range(1, len(edges)):
    m = idx == b
    if m.sum() >= 200:
        cen.append((edges[b - 1] + edges[b]) / 2); pwin.append(white_win[m].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.hist(np.clip(gap, -400, 400), bins=60, color="#8064a2")
ax1.set(xlabel="Rating gap (white - black)", ylabel="Games",
        title="Matchmaking pairs similar ratings")
ax1.text(0.02, 0.95, f"median |gap| = {np.median(np.abs(gap)):.0f} Elo",
         transform=ax1.transAxes, va="top", fontsize=10,
         bbox=dict(boxstyle="round", fc="white", ec="0.7"))
ax2.plot(cen, np.array(pwin) * 100, "-o", color="#c0504d", ms=4)
ax2.axhline(50, color="k", ls="--", lw=1)
ax2.set(xlabel="Rating gap (white - black)", ylabel="P(white wins) %",
        title="Result is highly predictable from the gap")
fig.tight_layout()
show(save_fig(fig, "eda_opponent_leak", FIG))

**Reading it.** Games cluster tightly around a zero rating gap (median |gap| well under 100 Elo), and
the win probability sweeps smoothly with the gap. Together these mean *opponent rating ≈ your rating*
and *result carries the gap*. That is why every instance drops the opponent's rating and username
(`clean.INSTANCE_COLUMNS`), why features describe **only the player's own play**, and why train/test
splits are grouped by `username`. The one place the opponent rating is reconstructed —
`model.add_opponent_rating` — exists solely to *measure* this leak as the "copy-opponent" baseline in
notebook 03, never to feed a model.

## 9. Feature ↔ rating correlation map

Finally, a bird's-eye view: how the informative features correlate with rating and with each other.

In [ ]:
# Curated set including opening_acc_mean (the strongest single correlate, from §7).
key = ["rating", "opening_acc_mean", "cpl_mean", "cpl_median", "acc_mean", "acc_after_book",
       "blunder_rate", "inaccuracy_rate", "cpl_std", "endgame_cpl_mean",
       "move_time_mean", "fast_move_share", "time_trouble_share", "n_moves", "reached_winning"]
corr = feats[key].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9.5, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.7}, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature <-> rating correlation")
show(save_fig(fig, "eda_correlation_heatmap", FIG))

print("Signed single-feature correlations with rating (sorted):")
print(corr["rating"].drop("rating").sort_values(key=abs, ascending=False).round(3).to_string())

**Reading it.** No single feature is a silver bullet. The strongest linear correlates cluster around
|r| ≈ 0.3–0.44: `opening_acc_mean` (+0.44) leads, followed by `cpl_mean` (−0.36), `acc_mean` (+0.31),
and `fast_move_share` (+0.31). The move-quality features are also strongly correlated *with each
other* — they are different views of "how many good moves did you find" — so their information
overlaps. The clock and style features add smaller, partly independent increments. A best-single-
correlation near 0.44 already caps what a *single game* can do linearly, which is why the model needs
the non-linear feature interactions that gradient-boosted trees provide (notebook 04), and why the
honest output is an interval, not a point.

## 10. The tension, in one chart

To make the signal-vs-noise story unmissable, here is a strongly rating-discriminative feature,
`cpl_mean`, shown as a distribution *within* each rating band. If a single game were enough, these
distributions would be well separated. They are not — they overlap heavily.

In [ ]:
order = [f"{BANDS[i]}-{BANDS[i+1]}" for i in range(len(BANDS) - 1)]
sub = feats[feats["cpl_mean"] < 120].copy()      # clip a thin high-CPL tail for readability
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.violinplot(data=sub, x="_band", y="cpl_mean", order=order, cut=0,
               palette="Blues", inner="quartile", ax=ax)
ax.set(xlabel="Rating band", ylabel="Mean centipawn loss (per game)",
       title="Strong trend in the means, heavy overlap in the distributions")
ax.tick_params(axis="x", rotation=20)
show(save_fig(fig, "eda_cpl_overlap_by_band", FIG))

## Takeaways

- **The signal is real.** Move quality (CPL, Accuracy%, opening accuracy, blunder rate) tracks rating
  strongly and monotonically. Chess strength genuinely leaves fingerprints in a single game.
- **The noise is larger.** Every one of those features has a wide, overlapping *within-band*
  distribution. One game pins a rating only loosely — the driving tension of the project.
- **Where the signal lives is not where you'd guess.** The opening (theory) separates players most;
  the endgame — later and "harder" — barely discriminates. This justifies per-phase features and the
  presence flags.
- **Selection bias and the single-game median of 1** frame what "success" can mean: a well-calibrated
  *interval* per game, and precision recovered by *aggregation* when more games exist.
- **The opponent is off-limits.** Matchmaking makes opponent rating a near-perfect but illegitimate
  predictor; the entire feature/splitting design is built to keep it out.

**Next:** notebook 02 shows exactly how a raw PGN becomes the 77-feature vector these plots were
built on — including the missing-value contract that lets a tree treat "no endgame happened" as
information rather than as a zero.